# LangChain 메모리 패턴 전환 — Legacy vs Modern 아키텍처 비교

## 이번 노트북 학습 목표

- Legacy 메모리 클래스(01~07번)와 Modern 패턴(08~14번)의 1:1 대응 관계를 이해해요.
- `RunnableWithMessageHistory`로 버퍼 메모리를 구현하고 세션 분리를 확인해요.
- 윈도우/토큰/요약/엔티티/그래프 메모리로 확장하는 설계 방향을 파악해요.

## 사전 지식

- `RunnableWithMessageHistory` 패턴 (10번 노트북)
- `ChatMessageHistory` 기본 사용법

## Legacy → Modern 아키텍처 매핑

> 🎯 **강의 포인트**: 이 노트북이 Ch04 전체의 핵심 브리지입니다. Legacy(01~07)와 Modern(08~14)의 1:1 매핑을 화이트보드에 그려가며 설명하면 효과적입니다. "클래스 하나"에서 "히스토리 + 미들웨어 + 스토어 조합"으로 분리된 것이 핵심입니다.

> 🔑 **핵심 개념**: Modern 패턴에서 메모리는 세 가지로 분리됩니다 — **저장소**(ChatMessageHistory), **전처리 로직**(윈도우/토큰/요약 미들웨어), **구조화 정보**(Store + Tool). 이 분리 덕분에 각 컴포넌트를 독립적으로 교체할 수 있습니다.

```mermaid
flowchart LR
    subgraph LEGACY["Legacy (01~07번)"]
        direction TB
        L1[ConversationBufferMemory]
        L2[ConversationBufferWindowMemory]
        L3[ConversationTokenBufferMemory]
        L4[ConversationSummaryMemory]
        L5[ConversationSummaryBufferMemory]
        L6[ConversationEntityMemory]
        L7[ConversationKGMemory]
    end

    subgraph MODERN["Modern (08~14번)"]
        direction TB
        M1[ChatMessageHistory<br/>+ RunnableWithMessageHistory]
        M2[ChatMessageHistory<br/>+ 윈도우 트리밍 미들웨어]
        M3[ChatMessageHistory<br/>+ 토큰 트리밍 미들웨어]
        M4[ChatMessageHistory<br/>+ 요약 체인]
        M5[ChatMessageHistory<br/>+ 요약 + 윈도우 조합]
        M6[InMemoryStore<br/>+ 엔티티 추출 Tool]
        M7[그래프 DB<br/>+ 관계 추출 Tool]
    end

    L1 --> M1
    L2 --> M2
    L3 --> M3
    L4 --> M4
    L5 --> M5
    L6 --> M6
    L7 --> M7

    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    class L1,L2,L3,L4,L5,L6,L7 storage
    class M1,M2,M3,M4,M5,M6,M7 process
```

핵심 변화: **"메모리 클래스 하나"**에서 **"히스토리 + 전처리 로직 + 스토어"** 조합으로 분리됐어요.

## 이 노트북의 위치: Legacy와 Modern을 잇는 다리(Bridge)

Ch04 메모리 챕터는 크게 세 구간으로 나뉘어요.

```
[01~07] Legacy 메모리        [11] 브리지(이 노트북)        [08~14] Modern 메모리
ConversationBufferMemory  ──────────────────────────>  ChatMessageHistory
ConversationChain         ──────────────────────────>  RunnableWithMessageHistory
memory.save_context()     ──────────────────────────>  invoke() 자동 저장
```

### 왜 전환이 필요했을까요?

Legacy 패턴의 한계:
1. **결합도가 높음**: 메모리 저장, 전처리, 체인 실행이 하나의 클래스에 묶여 있어요.
2. **다중 사용자 지원 어려움**: `session_id` 개념이 없어서 사용자별 메모리를 직접 관리해야 했어요.
3. **저장소 교체가 어려움**: SQLite에서 PostgreSQL로 바꾸려면 코드 전체를 수정해야 했어요.

Modern 패턴의 장점:
1. **관심사 분리**: 저장소(`ChatMessageHistory`), 전처리(윈도우/요약), 실행(`RunnableWithMessageHistory`)이 각각 독립적이에요.
2. **세션 기반 설계**: `session_id` 하나로 다중 사용자를 자연스럽게 지원해요.
3. **저장소 교체 용이**: `get_session_history` 함수만 바꾸면 돼요.

### Legacy → Modern 매핑 요약

| Legacy 클래스 (01~07번) | Modern 대체 패턴 (08~14번) | 핵심 변화 |
|------------------------|---------------------------|----------|
| `ConversationBufferMemory` | `ChatMessageHistory` + `RunnableWithMessageHistory` | 전체 대화 저장, 세션 자동 관리 |
| `ConversationBufferWindowMemory` | `ChatMessageHistory` + `last_k_messages()` 함수 | 윈도우 로직이 미들웨어로 분리 |
| `ConversationTokenBufferMemory` | `ChatMessageHistory` + `trim_messages()` 함수 | 토큰 트리밍이 미들웨어로 분리 |
| `ConversationSummaryMemory` | `ChatMessageHistory` + 요약 체인 (LLM 미들웨어) | 요약 로직이 별도 체인으로 분리 |
| `ConversationSummaryBufferMemory` | `ChatMessageHistory` + 요약 + 윈도우 조합 | 요약과 윈도우를 자유롭게 조합 |
| `ConversationEntityMemory` | `InMemoryStore` + 엔티티 추출 Tool | 구조화 정보가 Store+Tool로 분리 |
| `ConversationKGMemory` | 그래프 DB + 관계 추출 Tool | 지식그래프가 외부 DB+Tool로 분리 |

> 이 노트북을 통해 Legacy와 Modern의 연결고리를 이해한 뒤, 12번 이후 노트북에서 각 패턴을 실제 코드로 구현해요.

## 2. RunnableWithMessageHistory로 버퍼 메모리 구현

`ConversationBufferMemory` 없이도 **세션별 전체 대화 버퍼**를 구현할 수 있어요. 핵심 구성 요소 세 가지를 기억하세요.

- `ChatMessageHistory`(채팅 메시지 히스토리): 한 세션의 메시지 리스트를 관리해요.
- `RunnableWithMessageHistory`(실행 가능한 메시지 히스토리 래퍼): 체인에 "메모리 기능"을 덧입히는 래퍼예요.
- `session_id`(세션 식별자): 사용자를 구분하는 키예요 (쿠키·토큰·유저 ID 등과 연결 가능).

### 버퍼 메모리 구성 흐름

```mermaid
flowchart LR
    BASE_CHAIN["LCEL 기본 체인<br/>prompt | model | parser"] --> WRAP[RunnableWithMessageHistory<br/>래퍼로 감싸기]
    SESSION_FN[get_session_history<br/>session_id → ChatMessageHistory] --> WRAP
    WRAP --> BUFFER_CHAIN[buffer_chain<br/>invoke 시 자동 로드+저장]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class BASE_CHAIN process
    class SESSION_FN storage
    class WRAP process
    class BUFFER_CHAIN output
```

In [1]:
# ---------------------------------------------------
# 공통 설정: 임포트, 모델, 프롬프트, 세션 스토어 초기화
# ---------------------------------------------------

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# 1단계: LLM 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: 공통 프롬프트 (전체 노트북에서 재사용)
# - chat_history: RunnableWithMessageHistory가 자동으로 채워주는 변수
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# 3단계: 기본 체인 (메모리 없음)
base_chain = prompt | model | StrOutputParser()

# 4단계: 세션별 대화 기록 저장소
store: dict[str, ChatMessageHistory] = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    """session_id → ChatMessageHistory 반환 (없으면 새로 생성)."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 5단계: 전체 버퍼 메모리 체인 (RunnableWithMessageHistory 래퍼)
# - RunnableWithMessageHistory: invoke() 시 히스토리 자동 로드 + 저장
buffer_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [2]:
# ---------------------------------------------------
# 버퍼 메모리 동작 확인: 동일 세션에서 이전 대화 기억
# ---------------------------------------------------

# 1단계: config 설정 — session_id로 대화 세션을 지정
# - configurable 딕셔너리 안에 session_id를 넣어야 RunnableWithMessageHistory가 인식
config = {"configurable": {"session_id": "user_001"}}

# 2단계: 첫 번째 질문 (자기소개)
res1 = buffer_chain.invoke(
  {"question": "내 이름은 Elixir이야"},
  config=config
)
res1
print(f' ==> [Line 14]: \033[38;2;152;56;130m[res1]\033[0m({type(res1).__name__}) = \033[38;2;156;101;222m{res1}\033[0m')

# 3단계: 두 번째 질문 (이전 내용 기억 여부 확인)
res2 = buffer_chain.invoke(
  {"question": "내 이름이 뭐라고?"},
  config=config
)

res2
print(f' ==> [Line 23]: \033[38;2;249;126;129m[res2]\033[0m({type(res2).__name__}) = \033[38;2;94;17;122m{res2}\033[0m')


 ==> [Line 14]: [res1](str) = 안녕하세요, Elixir! 만나서 반갑습니다. 어떻게 도와드릴까요?
 ==> [Line 23]: [res2](str) = 당신의 이름은 Elixir입니다! 맞나요?


In [ ]:
# ---------------------------------------------------
# 버퍼 메모리 동작 확인: 다른 세션은 이전 내용을 알 수 없음
# ---------------------------------------------------

# 다른 session_id로 동일한 질문 → 이전 내용을 기억하지 못함
# - session_id가 다르면 완전히 독립된 대화 공간이 생성됨


In [4]:
# ============================================================
# TODO: 윈도우 메모리 체인을 구현하고 테스트하세요
# 힌트: build_window_chain(k_messages=6)으로 최근 k개 메시지만 사용하는 체인을 만드세요
# 예상 결과: 대화가 k개를 초과하면 오래된 내용은 잊어야 합니다
# ============================================================

# ---------------------------------------------------
# 윈도우 메모리 체인 구현 및 테스트
# ---------------------------------------------------

from typing import List
from langchain_core.messages import BaseMessage


def last_k_messages(messages: List[BaseMessage], k: int = 6) -> List[BaseMessage]:
    """대화 메시지 리스트에서 마지막 k개만 가져오기.

    - 한 턴(human + ai)을 2개 메시지로 보면, k=6이면 최근 3턴 정도에 해당합니다.
    """
    return messages[-k:]


# window_chain: 최근 k개 메시지만 프롬프트에 넣는 체인

def build_window_chain(k_messages: int = 6):
    """최근 k개 메시지만 사용하는 윈도우 메모리 체인을 만들어 반환"""

    def _chain(inputs: dict):
        session_id = inputs["session_id"]
        question = inputs["question"]
        history = get_session_history(session_id)
        # 1) 프롬프트 구성 (최근 k개만 사용)
        window = last_k_messages(history.messages, k=k_messages)
        msg = prompt.invoke({
            "chat_history": window,
            "question": question,
        })
        # 2) 모델 호출 + 문자열 파싱
        result = (model | StrOutputParser()).invoke(msg)
        # 3) 대화 기록에 현재 턴을 수동으로 추가
        history.add_user_message(question)
        history.add_ai_message(result)
        return result

    return _chain


window_chain = build_window_chain(k_messages=6)

# 동일 세션에서 여러 번 대화 → 내부적으로는 전체 히스토리를 저장하지만,
# 프롬프트에는 항상 최근 6개 메시지만 들어감
window_session_id = "window_user_001"

questions = [
    "1번째 질문입니다. 오늘 하루 계획을 같이 세워볼까요?",
    "2번째 질문입니다. 오전에는 어떤 일을 하면 좋을까요?",
    "3번째 질문입니다. 점심시간에는 뭘 먹을지 추천해 주세요.",
    "4번째 질문입니다. 오후에는 집중력을 유지하는 방법이 있을까요?",
    "5번째 질문입니다. 저녁에는 운동 계획을 세워보고 싶어요.",
]

answer = window_chain({"session_id": window_session_id, "question": questions[0]})
print(answer)
answer = window_chain({"session_id": window_session_id, "question": questions[1]})
print(answer)
answer = window_chain({"session_id": window_session_id, "question": questions[2]})
print(answer)


물론이죠! 오늘 하루 계획을 세우기 위해 몇 가지 질문을 드릴게요.

1. 오늘 어떤 일을 해야 하나요? (예: 업무, 공부, 운동 등)
2. 특별히 하고 싶은 활동이나 목표가 있나요?
3. 하루에 몇 시간 정도를 계획할 수 있나요?
4. 어떤 시간대에 가장 집중력이 좋으신가요?

이 정보를 바탕으로 계획을 세워보겠습니다!
오전 시간은 보통 집중력이 가장 높은 시간대이므로 중요한 일을 하는 것이 좋습니다. 다음은 오전에 할 수 있는 몇 가지 활동입니다:

1. **업무 또는 공부**: 가장 중요한 프로젝트나 과제를 시작하세요. 집중해서 처리할 수 있는 시간이므로, 방해받지 않도록 조용한 환경을 마련하는 것이 좋습니다.

2. **운동**: 가벼운 스트레칭이나 조깅, 요가 등을 통해 몸을 깨우고 에너지를 충전하세요. 운동 후에는 기분이 상쾌해지고 집중력이 높아질 수 있습니다.

3. **계획 세우기**: 하루의 목표를 정리하고 우선순위를 매기는 시간을 가져보세요. 오늘 해야 할 일들을 리스트로 작성하면 더 효율적으로 일할 수 있습니다.

4. **독서**: 관심 있는 책이나 자료를 읽으며 지식을 쌓는 것도 좋은 방법입니다. 특히 자기계발서나 전문 서적을 읽는 것이 도움이 될 수 있습니다.

5. **네트워킹**: 동료나 친구와의 커피 미팅을 통해 아이디어를 교환하거나 협업할 기회를 모색해보세요.

어떤 활동이 가장 마음에 드시나요? 또는 다른 아이디어가 있으신가요?
점심시간에 먹을 음식은 개인의 취향과 건강 상태에 따라 다르지만, 몇 가지 추천을 드릴게요:

1. **샐러드**: 신선한 채소와 단백질(닭가슴살, 두부, 계란 등)을 추가한 샐러드는 가볍고 건강한 선택입니다. 드레싱은 올리브 오일과 레몬즙으로 간단하게 만들어 보세요.

2. **비빔밥**: 다양한 채소와 고기를 넣고 고추장으로 비벼 먹는 비빔밥은 영양가가 높고 맛도 좋습니다. 취향에 따라 고추장을 조절할 수 있어요.

3. **파스타**: 토마토 소스나 크림 소스를 곁들인 파스타는 간편하면서도 맛있

## 4. 토큰 기반 메모리/요약 패턴 (ConversationTokenBufferMemory, ConversationSummaryMemory 대체 아이디어)

토큰 기반/요약 메모리는 LangChain 1.0에서는 **에이전트/그래프의 미들웨어**로 구현하는 것이 표준입니다.

이 노트북에서는 아이디어만 간단히 정리합니다.

### 4.1 토큰 트리밍 아이디어
- 전체 히스토리는 `ChatMessageHistory`에 쌓고,
- 모델 호출 직전에 `trim_messages()` 함수를 거쳐
  - 첫 시스템 메시지
  - 최근 N개의 human/AI 메시지
  만 남기고 나머지는 버립니다.

### 4.2 요약 기반 메모리 아이디어
- `SummarizationMiddleware` 또는 커스텀 요약 체인을 사용해
  - 오래된 대화는 **요약 텍스트**로 압축하고
  - 최근 대화는 원문 메시지로 유지합니다.
- 개념적으로는 `ConversationSummaryMemory` / `ConversationSummaryBufferMemory`와 동일하지만,
  - 구현은 **미들웨어 + 스토어** 조합으로 옮겨갔다고 이해하면 됩니다.

> 실제 코드는 LangChain 공식 문서의 `short-term-memory`, `SummarizationMiddleware` 섹션을 참고하여
> LangGraph 기반 에이전트 예제로 구성하는 것을 권장합니다.


## 5. 엔티티 / 지식그래프 메모리로 확장하는 방향

`ConversationEntityMemory`, `ConversationKGMemory`는 LangChain 1.0에서 **그대로 쓰기보다** 다음과 같이 재설계하는 것이 좋습니다.

- **엔티티 메모리**
  - 도구(tool) + LLM 체인으로 엔티티를 추출
  - `InMemoryStore` 또는 RDB/VectorStore 등에 `user_id` 기준으로 저장
  - 이후 대화에서 해당 스토어를 조회하는 도구를 만들어 사용

- **지식그래프 메모리**
  - 엔티티/관계 추출 체인 → 그래프 DB(예: Neo4j) 또는 자체 그래프 구조에 저장
  - 질의 시에는 그래프를 조회하는 도구를 통해 LLM에 컨텍스트 제공

즉, **"메모리"를 LangChain 1.0에서는**
- 단순 대화 이력 → `RunnableWithMessageHistory`
- 구조화된 정보(엔티티/관계/장기 정보) → `Store` + `Tool` + (선택적으로 LangGraph)
로 분리해서 설계하는 것이 현대적인 패턴입니다.

---

## 💡 이 노트북에서 가져가야 할 핵심

- **구식 메모리 클래스는 `langchain.memory`에 남겨두고,**
  - 교재에서는 "과거 방식"으로만 간단히 소개
- **실전 구현과 신식 교재는**
  - `RunnableWithMessageHistory` + `ChatMessageHistory`를 기본으로 삼고
  - 윈도우/토큰/요약/엔티티/그래프는 "별도 미들웨어/스토어/도구"로 분리해서 설명

이 11번 노트북을 "브리지"로 두고,
- 12번 이후 노트북에서 각각
  - "윈도우 메모리 구현", "토큰 기반 트리밍/요약", "엔티티/지식그래프 메모리"를
  - Runnable/Graph 패턴으로 자세히 풀어가면 좋습니다.

## 핵심 요약

### Legacy → Modern 메모리 패턴 매핑

| Legacy 클래스 | Modern 대체 패턴 |
|---------------|-----------------|
| `ConversationBufferMemory` | `ChatMessageHistory` + `RunnableWithMessageHistory` |
| `ConversationBufferWindowMemory` | `ChatMessageHistory` + `last_k_messages()` 윈도우 함수 |
| `ConversationTokenBufferMemory` | `ChatMessageHistory` + `trim_messages()` 토큰 트리밍 |
| `ConversationSummaryMemory` | `ChatMessageHistory` + 요약 체인(LLM 미들웨어) |
| `ConversationEntityMemory` | `InMemoryStore` + 엔티티 추출 Tool |
| `ConversationKGMemory` | 그래프 DB + 관계 추출 Tool |

### 설계 원칙

- **대화 이력** → `ChatMessageHistory`(또는 `SQLChatMessageHistory`)에 보관해요.
- **전처리 로직** → 모델 호출 직전 미들웨어(윈도우·토큰 트리밍·요약)로 분리해요.
- **구조화 정보** → `Store` + `Tool` 조합으로 별도 관리해요.
- **자동 저장** → `RunnableWithMessageHistory`가 `invoke()` 시 자동으로 처리해요.

### 주의사항

- `session_id`가 다르면 완전히 독립된 대화 공간이 생겨요. UUID 등 예측 불가능한 값을 사용하세요.
- 윈도우 패턴에서 오래된 메시지는 LLM에 전달되지 않지만, 히스토리에는 계속 쌓여요. 장기 운영 시 히스토리 정리 정책도 함께 설계하세요.
- Legacy 클래스(`ConversationBufferMemory` 등)는 0.2.x 이후 deprecated 상태예요. 신규 코드에는 Modern 패턴을 사용하세요.

### 다음 단계 예고

**12번**: `last_k_messages()`를 사용한 윈도우 메모리 패턴을 코드로 완성하고, 버퍼 전체 vs 윈도우를 직접 비교해요.

**13번**: 토큰 기반 트리밍과 요약 메모리 패턴을 구현해요.

**14번**: `InMemoryStore` + 엔티티 추출 Tool 조합으로 구조화된 사용자 정보를 저장하고 활용하는 패턴을 다뤄요.


## 전체 비교표: Legacy 7종 메모리 vs Modern 대체 패턴

아래 표는 Ch04에서 다루는 **모든 Legacy 메모리 클래스**와 그에 대응하는 **Modern 패턴**을 한눈에 정리한 것입니다.

| # | Legacy 클래스 | 노트북 | Modern 대체 패턴 | 노트북 | 핵심 구성 요소 | 프로덕션 적합성 |
|---|--------------|--------|------------------|--------|---------------|---------------|
| 1 | `ConversationBufferMemory` | 01번 | `ChatMessageHistory` + `RunnableWithMessageHistory` | 08, 10번 | 히스토리 저장소 + 래퍼 | 높음 (세션 자동 관리) |
| 2 | `ConversationBufferWindowMemory` | 02번 | `ChatMessageHistory` + `last_k_messages()` | 11, 12번 | 히스토리 + 윈도우 함수 | 높음 (토큰 절약) |
| 3 | `ConversationTokenBufferMemory` | 03번 | `ChatMessageHistory` + `trim_messages()` | 11, 13번 | 히스토리 + 토큰 트리밍 함수 | 높음 (정밀 토큰 제어) |
| 4 | `ConversationSummaryMemory` | 06번 | `ChatMessageHistory` + 요약 체인 (LLM) | 11, 13번 | 히스토리 + LLM 요약 미들웨어 | 높음 (장기 대화 압축) |
| 5 | `ConversationSummaryBufferMemory` | — | `ChatMessageHistory` + 요약 + 윈도우 조합 | 13번 | 히스토리 + 요약 + 윈도우 결합 | 높음 (가장 실용적) |
| 6 | `ConversationEntityMemory` | 04번 | `InMemoryStore` + 엔티티 추출 Tool | 14번 | 키-값 저장소 + LLM Tool | 중간 (별도 설계 필요) |
| 7 | `ConversationKGMemory` | 05번 | 그래프 DB (Neo4j 등) + 관계 추출 Tool | 14번 | 그래프 DB + LLM Tool | 중간 (인프라 필요) |

### 설계 철학의 변화

```
Legacy: 하나의 클래스가 [저장 + 전처리 + 조회]를 모두 담당
         ConversationBufferWindowMemory(k=5, llm=llm)

Modern: 세 가지 역할을 독립 컴포넌트로 분리
         저장소: ChatMessageHistory / SQLChatMessageHistory
         전처리: last_k_messages() / trim_messages() / 요약 체인
         통합:   RunnableWithMessageHistory (자동 로드 + 저장)
```

### 어떤 패턴을 선택해야 할까요?

| 상황 | 추천 패턴 | 이유 |
|------|----------|------|
| 짧은 대화 (5턴 이하) | 버퍼 (전체 저장) | 단순하고 정보 손실 없음 |
| 중간 대화 (5~20턴) | 윈도우 (`last_k_messages`) | 토큰 절약하면서 최근 맥락 유지 |
| 긴 대화 (20턴 이상) | 요약 + 윈도우 조합 | 오래된 내용은 요약, 최근 내용은 원문 |
| 사용자 프로필 저장 | 엔티티 추출 (`Store` + `Tool`) | 이름, 직업 등 구조화된 정보 장기 보관 |
| 복잡한 관계 추적 | 지식그래프 (`GraphDB` + `Tool`) | 사람-조직-관계 등 네트워크형 정보 |

> 대부분의 챗봇 서비스에서는 **윈도우 + 요약 조합**이 가장 실용적입니다. 12~13번 노트북에서 실제 구현을 다뤄요.